In [2]:
%cd ../
%load_ext autoreload
%autoreload 2

/Users/matthaei/Documents/code/python/bachelor-project


In [3]:
# BiLSTM training for 12-hour wind vector forecasting

# This notebook trains `BiLSTMModel` on 10-min measurements to forecast the next 12 hours (72 steps)
# of u/v vector components per station.

import os
from omegaconf import OmegaConf
from loguru import logger
import pandas as pd

from src.database.database_service import DatabaseService
from src.weather_stations.weather_station_service import WeatherStationService
from src.measurements.measurement_service import MeasurementService
from src.model.variant.bilstm_model import BiLSTMModel


/Users/matthaei/Documents/code/python/bachelor-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Load config and initialize services
cfg = OmegaConf.load("conf/config.yaml")

db = DatabaseService(cfg)
ws_service = WeatherStationService(cfg, db)
weather_stations = ws_service.load_from_database(only_relevant=True)

ms_service = MeasurementService(cfg, db, weather_stations)

logger.info(f"Loaded {len(weather_stations)} weather stations")


2025-09-13 11:47:06.910 | INFO     | src.weather_stations.weather_station_data_provider:load_from_database:228 - Loaded 50 weather stations from database
2025-09-13 11:47:06.937 | INFO     | __main__:<module>:10 - Loaded 50 weather stations


In [4]:
# Load measurements from DB
measurements_df = ms_service.load_all_measurements_from_database()
logger.info(f"Measurements: {len(measurements_df)} rows")


2025-09-13 11:39:44.866 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 1000000)
2025-09-13 11:40:00.000 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 2000000)
2025-09-13 11:40:14.901 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 3000000)
2025-09-13 11:40:31.836 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 4000000)
2025-09-13 11:40:47.062 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 5000000)
2025-09-13 11:41:04.336 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loa

In [5]:
measurements_df.to_parquet("data/measurements.parquet")

In [7]:
measurements_df = pd.read_parquet("data/measurements.parquet")

In [8]:
# Optional: downsample or filter timeframe for quicker training
end_date = pd.Timestamp('2025-04-01')
train_df = measurements_df[(measurements_df['record_date']<=end_date)].copy()

test_df = measurements_df[measurements_df['record_date']>end_date].copy()

train_df.head()

,air_pressure,air_temperature_2m,air_temperature_5cm,average_wind_direction,average_wind_speed,dew_point_temperature,id,precipitation_duration,precipitation_indicator,quality_level,record_date,relative_humidity,station_id,sum_precipitation_height
0,-999.0,26.8,31.2,-999.0,-999.0,10.4,8676072,-999.0,-999,-999,2021-06-27 13:40:00,35.9,3426,0.0
1,-999.0,26.8,31.6,-999.0,-999.0,9.8,8676073,-999.0,-999,-999,2021-06-27 13:50:00,34.5,3426,0.0
2,-999.0,27.0,31.4,-999.0,-999.0,9.9,8676074,-999.0,-999,-999,2021-06-27 14:00:00,34.2,3426,0.0
3,-999.0,26.9,31.0,-999.0,-999.0,10.3,8676075,-999.0,-999,-999,2021-06-27 14:10:00,35.3,3426,0.0
4,-999.0,27.3,31.7,-999.0,-999.0,9.6,8676076,-999.0,-999,-999,2021-06-27 14:20:00,33.0,3426,0.0


In [ ]:
# Instantiate BiLSTM model
model = BiLSTMModel(
    history_steps=72,   # 12 hours of history
    horizon_steps=72,   # 12 hours forecast
    station_embedding_dim=8,
    hidden_size=128,
    num_layers=2,
    dropout=0.1,
    batch_size=64,
    learning_rate=1e-3,
    num_epochs=5, 
    val_split=0.2, 
    early_stopping_patience=5, 
    early_stopping_min_delta=5e-5, 
    restore_best_weights=True  # increase for real training
)
model


In [46]:
# Train model
model.train(train_df)


2025-09-12 16:27:50.207 | INFO     | src.model.variant.bilstm_model:train:15 - Preparing training: train_sequences=3970761, val_sequences=991046, stations=48, feature_dim=10, device=cuda
2025-09-12 16:27:50.220 | INFO     | src.model.variant.bilstm_model:train:18 - Model initialized: hidden_size=128, num_layers=2, dropout=0.1, params=650,000
2025-09-12 16:27:50.220 | INFO     | src.model.variant.bilstm_model:train:26 - DataLoader ready: train_samples=3970761, val_samples=991046, batch_size=64, batches_per_epoch=62044
2025-09-12 16:27:50.220 | INFO     | src.model.variant.bilstm_model:train:30 - CUDA available: using GPU 'NVIDIA GeForce RTX 4080 SUPER'
2025-09-12 16:27:50.221 | INFO     | src.model.variant.bilstm_model:train:40 - Epoch 1/5 starting (lr=1.000e-03)
2025-09-12 16:28:06.359 | INFO     | src.model.variant.bilstm_model:train:55 - Epoch 1 [6204/62044] batch_loss=0.000171
2025-09-12 16:28:22.359 | INFO     | src.model.variant.bilstm_model:train:55 - Epoch 1 [12408/62044] batch_

In [47]:
# Save model
save_dir = "models/new_model"
os.makedirs(save_dir, exist_ok=True)
model.save(save_dir)
logger.info(f"Saved BiLSTM model to {save_dir}")


2025-09-12 16:43:06.659 | INFO     | __main__:<module>:5 - Saved BiLSTM model to models/new_model


In [9]:
new_model = BiLSTMModel()

In [10]:
new_model.load('models/good_lstm')

In [13]:
# Quick prediction demo for a subset (uses last 72 steps per station)
preds = new_model.predict(measurements_df)
preds.head()


,station_id,record_date,u_pred,v_pred
0,96,2025-09-10 08:30:00,-0.483375,-0.562221
1,96,2025-09-10 08:40:00,-0.478859,-0.562072
2,96,2025-09-10 08:50:00,-0.481727,-0.563396
3,96,2025-09-10 09:00:00,-0.486213,-0.564773
4,96,2025-09-10 09:10:00,-0.493080,-0.566646


In [15]:
evaluation = new_model.evaluate(test_df)


2025-09-13 11:52:27.897 | INFO     | src.model.variant.bilstm_model:evaluate:14 - Evaluation: samples=1112291, batch_size=64, batches=17380
2025-09-13 11:56:41.836 | INFO     | src.model.variant.bilstm_model:evaluate:63 - Evaluation metrics: mae_u=0.1071, rmse_u=0.2863, mae_v=0.0641, rmse_v=0.2913, mae_speed=0.0651, rmse_speed=0.1736, mae_direction_deg=6.35°


In [16]:
per_h = new_model.evaluate_per_horizon(test_df, save_dir="artifacts/metrics2")

2025-09-13 12:00:57.337 | INFO     | src.model.variant.bilstm_model:evaluate_per_horizon:72 - Computed per-horizon metrics for 1..%d steps
2025-09-13 12:00:57.824 | INFO     | src.model.variant.bilstm_model:evaluate_per_horizon:104 - Saved per-horizon plots to /Users/matthaei/Documents/code/python/bachelor-project/artifacts/metrics2


In [21]:
per_h = model.evaluate_per_horizon(test_df)

2025-09-12 15:19:12.368 | INFO     | src.model.variant.bilstm_model:evaluate_per_horizon:71 - Computed per-horizon metrics for 1..%d steps
